# 02b — Encounter features

Compute previous encounters and time since the last admission. Duplicate spells are reduced before feature generation.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.io import load_pkg_yaml

schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_first_spell(df: pd.DataFrame, name: str = "episodes") -> pd.DataFrame:
    """Keep the first source row for each SUBJECT/SPELL_IDENTIFIER pair."""
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(KEYS, keep="first").reset_index(drop=True)
    print(f"{name}: {before:,} rows -> {len(result):,} unique spells")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/notebooks/data/synthetic/10000p


In [4]:
from icare_risk.clinphen.utils import (
    compute_previous_encounters,
    compute_time_since_last_admission
)

df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
df_episodes = keep_first_spell(df_episodes)
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

df_encounters = compute_previous_encounters(df_episodes, days=365)
df_encounters = compute_time_since_last_admission(
    df_episodes=df_encounters,
    subject_col="SUBJECT",
    spell_col="SPELL_IDENTIFIER",
    admission_col="ADMISSION_DATE",
    discharge_col="DISCHARGE_DATE",
    use_discharge_time=True,
)
df_encounters = keep_first_spell(df_encounters, "encounter features")
display(df_encounters.head())


episodes: 10,000 rows -> 10,000 unique spells
encounter features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,index_admission,num_encounters_last_365d,days_since_last_discharge
0,10001,7929089,2005940,4725136,2020-06-10,22:22:00,35,2020-07-03,320,cardiology,7,2020-06-10 22:22:00,0,NaN
1,10002,8200425,6570033,6896867,2020-03-27,17:31:00,63,2020-03-28,301,gastroenterology,6,2020-03-27 17:31:00,0,NaN
2,10003,9931252,5492977,1426665,2023-01-06,00:18:00,89,2023-01-25,301,gastroenterology,5,2023-01-06 00:18:00,0,NaN
3,10004,5095152,3730006,3105590,2022-09-14,10:32:00,25,2022-10-12,301,gastroenterology,4,2022-09-14 10:32:00,0,NaN
4,10005,7807609,7413177,1104787,2023-09-08,10:55:00,80,2023-09-23,140,oral surgery,none,2023-09-08 10:55:00,0,NaN


In [5]:
output_path = DATA_DIR / "02c_derived_features_encounters.parquet"
df_encounters.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/notebooks/data/synthetic/10000p/02c_derived_features_encounters.parquet
